# Step 2: Process Features

## Traitement des attributs

In [1]:
%load_ext autoreload
%autoreload 2


import geopandas as gpd
import pandas as pd
from functools import reduce
import osmnx as ox
from shapely.geometry import Point
from matplotlib import pyplot as plt

import os

from method_a_buffer import extract_buffer_feature
from feature_query import make_feature_query

import geopandas as gpd
import folium
from folium import Choropleth, CircleMarker, GeoJson
import osmnx as ox
import branca.colormap as cm
import rasterio
# Display the map in the notebook
from IPython.display import display
# Display all columns in a dataframe
pd.set_option('display.max_columns', None)


operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

input_file_path = '../../Data/input'
output_step1_path='../../Data/output/step-1'
output_step2_path='../../Data/output/step-2'
output_step3_path='../../Data/output/step-3'


# Load segments GeoDataFrame (with 'segment_id')
print("Loading pedestrian segments...")
# reLoad pedestrian segments
segmented_net = gpd.read_parquet(os.path.join(output_step1_path, "step1_pedestrian_segments.parquet"))
segmented_net["geometry"] = segmented_net["geometry"].apply(lambda geom: geom.buffer(0) if not geom.is_valid else geom)
segmented_net = segmented_net.to_crs(operation_crs)

# Import des attributs
attributs_info = pd.read_excel(f"{input_file_path}/attributs/attributs_info.xlsx", sheet_name="attributs_info")



Loading pedestrian segments...


In [2]:

# Garder uniquement les colonnes 'segment_id', 'vitesse', et 'geometry' dans segmented_net
segmented_net = segmented_net[['osmid', 'maxspeed', 'geometry', 'segment_id', 'length']].copy()
print('Boucle sur chaque attribut... peut prendre du temps (15-20mn)')

# Boucle sur le derniers attributs
for _, row in attributs_info.iterrows():
    if row['include_in_index']:
        attribute_name = row['attribute']
        method = row['method']
        how = row['how']
        value_column = row['value_column']
        buffer_size = row['buffer_size']
        geometry_type = row['geometry_type']
        feature_query_expr = make_feature_query(
        row.get('filter_column'),
        row.get('filter_values')
    )

        # Charger la couche attribut depuis gpd_attributs
        attribute_gdf = gpd.read_parquet(f"../../Data/output/step-2/parquet_attributs/{attribute_name}.parquet")
        attribute_gdf = attribute_gdf.to_crs(segmented_net.crs)

        # Appliquer la méthode
        if method == "A": # Buffer feature extraction
            attribute_df = extract_buffer_feature(
                segments_gdf = segmented_net,
                feature_gdf = attribute_gdf,
                feature_name=attribute_name,
                geom_kind=geometry_type,
                buffer_radius=buffer_size,
                how=how,
                value_column=value_column,
                crs_meter_epsg=operation_crs,
                feature_query  = feature_query_expr,
            )
            print(f"Buffer feature extracted for {attribute_name} with method {method}")
        
                # Debug duplicates
            if attribute_df[attribute_df.duplicated('segment_id')].shape[0] > 0:
                print("\nDEBUG: Found duplicate segment assignments")
                dupes = attribute_df[attribute_df.duplicated('segment_id', keep=False)]
                print(f"Number of segments with multiple zones: {len(dupes['segment_id'].unique())}")
            
                # Keep only the first occurrence for each segment_id
                attribute_df = attribute_df.drop_duplicates('segment_id', keep='first')
                print("Dropped duplicates, keeping first occurrence")
            print(f"Spatial join computed for {attribute_name} with method {method}") 
        
        # Ajoute d'autres méthodes si besoin


        # Ajouter la colonne au GeoDataFrame principal
        segmented_net[f'{attribute_name}_{method}_{buffer_size}'] = attribute_df[attribute_name].fillna(0)

# Sauvegarder
segmented_net.to_crs(target_crs).to_parquet(os.path.join(output_step2_path, "step2_features.parquet"), index=False)


Boucle sur chaque attribut... peut prendre du temps (15-20mn)
Buffer feature extracted for arbre_isole with method A
Spatial join computed for arbre_isole with method A
Buffer feature extracted for espace_vert with method A
Spatial join computed for espace_vert with method A
Buffer feature extracted for accident with method A
Spatial join computed for accident with method A
Buffer feature extracted for zone_apaisee with method A
Spatial join computed for zone_apaisee with method A
Buffer feature extracted for zone_pietonne with method A
Spatial join computed for zone_pietonne with method A
Buffer feature extracted for vitesse with method A
Spatial join computed for vitesse with method A
Buffer feature extracted for ratio_trottoir with method A
Spatial join computed for ratio_trottoir with method A
Buffer feature extracted for eau with method A
Spatial join computed for eau with method A
Buffer feature extracted for rez_actif with method A
Spatial join computed for rez_actif with method

In [3]:

segmented_net.head(20)


,osmid,maxspeed,geometry,segment_id,length,arbre_isole_A_10,espace_vert_A_10,accident_A_10,zone_apaisee_A_10,zone_pietonne_A_10,vitesse_A_10,ratio_trottoir_A_10,eau_A_10,rez_actif_A_10,stationnement_genant_A_10,bruit_A_10,tp_A_10,amenite_A_10,espaces_ouverts_A_10,temperature_A_10
0,680474053,None,"LINESTRING (2501551.092 1118168.667, 2501553.0...",680474053_001,7.236785,0.0,0.083783,2.0,1.831794,0.000000,1.831794,0.240748,0.0,0.0,12.0,0.0,0.0,0.0,0.0,31.744839
1,680474053,None,"LINESTRING (2501551.092 1118168.667, 2501548.2...",680474053_002,10.533514,0.0,0.005835,2.0,2.414139,0.000000,2.414139,0.538427,0.0,0.0,1.0,0.0,0.0,0.0,1.0,32.526081
2,678844695,None,"LINESTRING (2502265.747 1118589.306, 2502270.5...",678844695_001,9.102543,12.0,0.059789,0.0,3.099900,0.000000,3.099900,0.208070,0.0,0.0,0.0,0.0,1.0,0.0,1.0,29.621059
3,678844695,None,"LINESTRING (2502265.747 1118589.306, 2502259.1...",678844695_002,11.727643,10.0,0.091713,1.0,1.939113,0.000000,1.939113,0.644849,1.0,0.0,0.0,0.0,0.0,0.0,1.0,28.879360
4,357504864,None,"LINESTRING (2502914.784 1119660.96, 2502908.91...",357504864_001,6.640850,8.0,0.017871,4.0,5.044367,0.000000,5.044367,0.141975,0.0,0.0,0.0,0.0,0.0,0.0,0.0,30.659426
5,357504864,None,"LINESTRING (2502914.784 1119660.96, 2502917.04...",357504864_002,4.515569,0.0,0.000000,4.0,6.413578,0.000000,6.413578,0.176408,0.0,0.0,0.0,1.0,0.0,0.0,0.0,31.264500
6,1347981918,None,"LINESTRING (2502914.784 1119660.96, 2502917.04...",1347981918_001,4.515569,0.0,0.000000,4.0,6.413578,0.000000,6.413578,0.176408,0.0,0.0,0.0,1.0,0.0,0.0,0.0,31.264500
7,813855859,None,"LINESTRING (2497091.678 1119566.278, 2497096.9...",813855859_001,5.331805,0.0,0.000000,0.0,17.157335,0.000000,10.980695,0.231175,0.0,0.0,0.0,0.0,0.0,0.0,0.0,33.335823
8,813855859,None,"LINESTRING (2497091.678 1119566.278, 2497087.0...",813855859_002,4.729008,0.0,0.000000,0.0,12.586365,0.000000,8.055273,0.104425,0.0,0.0,0.0,0.0,0.0,0.0,0.0,33.345776
9,132523795,None,"LINESTRING (2499924.593 1116521.38, 2499916.03...",132523795_001,8.749742,0.0,0.000573,0.0,2.550162,0.000000,0.283351,0.284289,0.0,0.0,0.0,0.0,0.0,0.0,0.0,30.625126


In [4]:
#import step2_features and convert to gpkg for QGIS use
segmented_net = gpd.read_parquet(os.path.join(output_step2_path, "step2_features.parquet"))
segmented_net.to_file(os.path.join(output_step2_path, "step2_features.gpkg"), driver="GPKG")
